# Main ingestion pipeline: PDF to Qdrant vector store

# Import

In [41]:
import torch
import pypdfium2 as pdfium
import time
import json
import re
from collections import Counter
from unstructured.partition.md import partition_md
from unstructured.documents.elements import Title
from unstructured.chunking.title import chunk_by_title

# Inference

In [42]:
import pypdfium2 as pdfium

PDF_PATH = "./benchmark_pdfs/1706.03762.pdf"
pdf = pdfium.PdfDocument(PDF_PATH)
images = [page.render(scale=1540/max(page.get_width(), page.get_height())).to_pil() for page in pdf]
print(f"{len(images)} pages")

15 pages


In [43]:
# !pip install aiohttp
import asyncio
import aiohttp
import base64
import time
import logging
from io import BytesIO
from typing import List
from PIL import Image

# Configure basic logging for the cell
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

VLLM_API_URL = "http://localhost:8000/v1/chat/completions"
MODEL_NAME = "lighton-ocr-2-1b"

# Semaphore to control max concurrent HTTP connections. 
# Set this close to your vLLM max_num_seqs to keep the engine saturated without HTTP timeouts.
MAX_CONCURRENT_REQUESTS = 16 

def encode_image_base64(img: Image.Image) -> str:
    """
    Converts a PIL Image to a base64 string.
    Using PNG format to avoid JPEG compression artifacts that degrade OCR accuracy.
    """
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

async def process_single_page(
    session: aiohttp.ClientSession, 
    img: Image.Image, 
    sem: asyncio.Semaphore, 
    page_index: int
) -> str:
    """
    Sends a single image to the vLLM server concurrently using the OpenAI Vision API format.
    """
    async with sem:
        img_b64 = encode_image_base64(img)
        
        # Standard OpenAI Chat Completion payload for Vision models
        payload = {
            "model": MODEL_NAME,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{img_b64}"}
                        }
                    ]
                }
            ],
            "max_tokens": 1024,
            "temperature": 0.0  # Greedy decoding is mathematically optimal for OCR tasks
        }
        
        try:
            async with session.post(VLLM_API_URL, json=payload) as response:
                response.raise_for_status()
                result = await response.json()
                return result["choices"][0]["message"]["content"]
        except Exception as e:
            logger.error(f"Failed to process page {page_index}: {str(e)}")
            return "" # Graceful degradation: return empty string on failure

async def run_vllm_inference(images: List[Image.Image]) -> List[str]:
    """
    Orchestrates the asynchronous execution of all OCR requests.
    """
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    
    # Custom TCPConnector to avoid bottlenecking on local socket limits
    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT_REQUESTS)
    
    async with aiohttp.ClientSession(connector=connector) as session:
        # Create a list of async tasks
        tasks = [
            process_single_page(session, img, sem, idx) 
            for idx, img in enumerate(images)
        ]
        
        # asyncio.gather preserves the original order of the iterable
        return await asyncio.gather(*tasks)

# --- Execution Block ---
# Note: Jupyter automatically handles top-level 'await'.
print(f"Starting async vLLM inference for {len(images)} pages...")
t0 = time.time()

results = await run_vllm_inference(images)

total_time = time.time() - t0
print(f"Total time: {total_time:.2f}s for {len(images)} pages.")
print(f"Throughput: {total_time / len(images):.3f} s/page | {len(images) / total_time:.2f} pages/s")

Starting async vLLM inference for 15 pages...
Total time: 8.14s for 15 pages.
Throughput: 0.543 s/page | 1.84 pages/s


In [44]:
results[14]

'The Law will never be perfect but its application should be just - this is what we are missing in my opinion <EOS> <pad>\n\nThe Law will never be perfect but its application should be just - this is what we are missing in my opinion <EOS> <pad>\n\nFigure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the sentence. We give two such examples above, from two different heads from the encoder self-attention at layer 5 of 6. The heads clearly learned to perform different tasks.\n\n15'

# Chunking

In [56]:
import os
import re
from collections import Counter
from typing import List, Dict, Tuple, Any, Optional

# --- Enforce Strict Offline Mode ---
# Prevents HuggingFace Hub from trying to verify models online
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

from transformers import AutoTokenizer
from chonkie import RecursiveChunker, RecursiveLevel, RecursiveRules

# --- Configuration ---
# Update this path to point to your upgraded local embedding model
# Recommendation: BAAI/bge-m3 or nomic-ai/nomic-embed-text-v1.5
MODEL_PATH = os.path.expanduser("~/AI_projects/chatbot/code/chunk/all-MiniLM-L6-v2")
CHUNK_SIZE_TOKENS = 512 # Upgraded size to prevent destroying HTML tables
OVERLAP_TOKENS = 64

# ==========================================
# 1. HELPER FUNCTIONS (Pure Python, No Unstructured)
# ==========================================

def _is_noise_chunk(text: str) -> bool:
    """Detects pagination artifacts, pure numbers, or OCR garbage."""
    tokens = re.findall(r"\b\w+\b", text or "")
    if not tokens:
        return True
    alpha = sum(any(c.isalpha() for c in t) for t in tokens)
    numeric = sum(t.isdigit() for t in tokens)
    return alpha < 3 or numeric > alpha * 3

def _build_page_spans(markdown_pages: List[str], separator: str = "\n\n") -> Tuple[str, List[Tuple[int, int, int]]]:
    """
    Concatenates pages and tracks exact character indices for each page.
    Returns: (full_text, [(page_number, start_char_index, end_char_index), ...])
    """
    spans = []
    current_pos = 0
    
    for i, page_text in enumerate(markdown_pages):
        if i > 0:
            current_pos += len(separator)
            
        start_idx = current_pos
        current_pos += len(page_text)
        end_idx = current_pos
        
        # 1-based page index
        spans.append((i + 1, start_idx, end_idx))
        
    full_text = separator.join(markdown_pages)
    return full_text, spans

def _extract_markdown_headers(full_text: str) -> List[Dict[str, Any]]:
    """
    Extracts all markdown headers (#, ##, etc.) with their character position.
    This replaces the heavy 'unstructured' library.
    """
    headers = []
    # Matches "# Title", "## Subtitle", etc.
    pattern = re.compile(r'^(#{1,6})\s+(.+)$', re.MULTILINE)
    
    for match in pattern.finditer(full_text):
        headers.append({
            'start_idx': match.start(),
            'level': len(match.group(1)),
            'text': match.group(2).strip()
        })
    return headers

def _get_section_path(chunk_start_idx: int, headers: List[Dict[str, Any]]) -> List[str]:
    """
    Rebuilds the hierarchical section path (e.g., ['1. Intro', '1.2 Details'])
    for a chunk based on its starting character index.
    """
    stack = []
    for h in headers:
        # Stop looking if the header appears AFTER the chunk starts
        if h['start_idx'] > chunk_start_idx:
            break
            
        # Pop headers from the stack that are at the same or deeper level
        while stack and stack[-1]['level'] >= h['level']:
            stack.pop()
            
        stack.append(h)
        
    return [h['text'] for h in stack]

def _get_page_range(chunk_start: int, chunk_end: int, page_spans: List[Tuple[int, int, int]]) -> Tuple[Optional[int], Optional[int]]:
    """Finds which pages the chunk overlaps with."""
    overlapping_pages = [
        page_num for page_num, p_start, p_end in page_spans
        if chunk_end > p_start and chunk_start < p_end
    ]
    if not overlapping_pages:
        return None, None
    return min(overlapping_pages), max(overlapping_pages)

def _detect_elements(text: str) -> List[str]:
    """Lightweight detection of tables and images based on markdown syntax."""
    elements = ["Text"]
    if re.search(r'\|.*\|.*\n\|[-:\s|]+\|', text) or "<table" in text.lower():
        elements.append("Table")
    if "![" in text or "<img" in text.lower():
        elements.append("Image")
    if "$$" in text or r"\begin{" in text:
        elements.append("Math")
    return elements

# ==========================================
# 2. MAIN OFFLINE PIPELINE
# ==========================================

def process_ocr_to_rag_chunks(doc_id: str, markdown_pages: List[str]) -> List[Dict[str, Any]]:
    """
    Main function to process OCR markdown pages into enriched RAG chunks.
    Runs entirely offline using Chonkie.
    """
    print(f"[{doc_id}] Building global text map...")
    
    # 1. Build spatial map and extract hierarchy (0 dependencies)
    full_text, page_spans = _build_page_spans(markdown_pages)
    headers = _extract_markdown_headers(full_text)
    
    # 2. Initialize Chonkie Offline
    print(f"[{doc_id}] Loading local tokenizer ({MODEL_PATH})...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    # Define hardcoded, safe Markdown rules (No online JSON recipe needed)
    # This prevents splitting inside tables or math blocks by prioritizing structural breaks
    offline_rules = [
        RecursiveLevel(delimiters=["\n# ", "\n## ", "\n### "], include_delim="next"),
        RecursiveLevel(delimiters=["\n\n"]),
        RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),
        RecursiveLevel(whitespace=True)
    ]

    # Build rules using the exact class used internally by RecursiveChunker.
    # This avoids class-identity mismatches in long-lived notebook kernels.
    from chonkie.types.recursive import RecursiveRules as ChonkieRecursiveRules
    rules_obj = ChonkieRecursiveRules(levels=offline_rules)

    chunker = RecursiveChunker(
        tokenizer=tokenizer,
        chunk_size=CHUNK_SIZE_TOKENS,
        rules=rules_obj
    )
    
    print(f"[{doc_id}] Chunking document (SIMD Accelerated)...")
    chonkie_chunks = chunker(full_text)
    
    # 3. Enrich chunks with spatial and hierarchical metadata
    raw_chunks = []
    for i, chunk in enumerate(chonkie_chunks, 1):
        cs, ce = chunk.start_index, chunk.end_index
        text = chunk.text.strip()
        
        # Apply noise filter early
        if _is_noise_chunk(text):
            continue
            
        page_start, page_end = _get_page_range(cs, ce, page_spans)
        section_path = _get_section_path(cs, headers)
        elements = _detect_elements(text)
        
        raw_chunks.append({
            "text": text,
            "metadata": {
                "doc_id": doc_id,
                "chonkie_token_count": chunk.token_count,
                "section_title": section_path[-1] if section_path else None,
                "section_path": section_path,
                "section_depth": len(section_path),
                "page_start": page_start,
                "page_end": page_end,
                "chunk_type": "Table" if "Table" in elements else "Text",
                "has_table": "Table" in elements,
                "has_image": "Image" in elements,
                "has_math": "Math" in elements,
                "char_len": len(text)
            }
        })
        
    # 4. Cleanup: Merge excessively short chunks to preserve semantic density
    cleaned_chunks = []
    for row in raw_chunks:
        if cleaned_chunks and len(row["text"]) < 260: # Threshold for short chunks
            prev = cleaned_chunks[-1]
            prev["text"] = f"{prev['text']}\n\n{row['text']}"
            
            # Merge metadata safely
            pm, cm = prev["metadata"], row["metadata"]
            starts = [x for x in (pm.get("page_start"), cm.get("page_start")) if x is not None]
            ends = [x for x in (pm.get("page_end"), cm.get("page_end")) if x is not None]
            
            pm["page_start"] = min(starts) if starts else None
            pm["page_end"] = max(ends) if ends else None
            pm["has_table"] = pm.get("has_table") or cm.get("has_table")
            pm["has_math"] = pm.get("has_math") or cm.get("has_math")
            pm["chonkie_token_count"] += cm["chonkie_token_count"] # Approximation after merge
            pm["char_len"] += cm["char_len"]
        else:
            cleaned_chunks.append(row)
            
    # 5. Finalize ID chain and positioning metrics
    total_chunks = len(cleaned_chunks)
    for idx, row in enumerate(cleaned_chunks, 1):
        m = row["metadata"]
        row["id"] = f"{doc_id}_ck{idx}"
        m["chunk_index"] = idx
        m["total_chunks"] = total_chunks
        m["position_ratio"] = round(idx / total_chunks, 3) if total_chunks else None
        m["prev_chunk_id"] = f"{doc_id}_ck{idx - 1}" if idx > 1 else None
        m["next_chunk_id"] = f"{doc_id}_ck{idx + 1}" if idx < total_chunks else None

    print(f"[{doc_id}] Finished! {len(chonkie_chunks)} raw chunks -> {len(cleaned_chunks)} refined chunks.")
    return cleaned_chunks

In [57]:
pdf_filename = "attention_is_all_you_need.pdf"
final_rag_chunks = process_ocr_to_rag_chunks(pdf_filename, results)
print(final_rag_chunks[0])

Token indices sequence length is longer than the specified maximum sequence length for this model (553 > 512). Running this sequence through the model will result in indexing errors


[attention_is_all_you_need.pdf] Building global text map...
[attention_is_all_you_need.pdf] Loading local tokenizer (/home/calla/AI_projects/chatbot/code/chunk/all-MiniLM-L6-v2)...
[attention_is_all_you_need.pdf] Chunking document (SIMD Accelerated)...
[attention_is_all_you_need.pdf] Finished! 32 raw chunks -> 30 refined chunks.
{'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\n---\n\n# Attention Is All You Need\n\n<table>\n  <tr>\n    <td>Ashish Vaswani*<br>Google Brain<br>avaswani@google.com</td>\n    <td>Noam Shazeer*<br>Google Brain<br>noam@google.com</td>\n    <td>Niki Parmar*<br>Google Research<br>nikip@google.com</td>\n    <td>Jakob Uszkoreit*<br>Google Research<br>usz@google.com</td>\n  </tr>\n  <tr>\n    <td>Llion Jones*<br>Google Research<br>llion@google.com</td>\n    <td>Aidan N. Gomez* †<br>University of Toronto<br>aidan@cs.toronto.edu</td

In [58]:
final_rag_chunks

[{'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\n---\n\n# Attention Is All You Need\n\n<table>\n  <tr>\n    <td>Ashish Vaswani*<br>Google Brain<br>avaswani@google.com</td>\n    <td>Noam Shazeer*<br>Google Brain<br>noam@google.com</td>\n    <td>Niki Parmar*<br>Google Research<br>nikip@google.com</td>\n    <td>Jakob Uszkoreit*<br>Google Research<br>usz@google.com</td>\n  </tr>\n  <tr>\n    <td>Llion Jones*<br>Google Research<br>llion@google.com</td>\n    <td>Aidan N. Gomez* †<br>University of Toronto<br>aidan@cs.toronto.edu</td>\n    <td>Łukasz Kaiser*<br>Google Brain<br>lukaszkaiser@google.com</td>\n  </tr>\n  <tr>\n    <td colspan="3">Illia Polosukhin* ‡<br>illia.polosukhin@gmail.com</td>\n  </tr>\n</table>',
  'metadata': {'doc_id': 'attention_is_all_you_need.pdf',
   'chonkie_token_count': 297,
   'section_title': None,
   'section_path': [],
   '

# Insertion qdrant

In [48]:
import json
import os

def save_chunks(chunks, filename="final_chunks.json"):
    """Saves the list of chunks to a JSON file."""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    print(f"Successfully saved {len(chunks)} chunks to {filename}")

def load_chunks(filename="final_chunks.json"):
    """Loads the list of chunks from a JSON file if it exists."""
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            chunks = json.load(f)
        print(f"Successfully loaded {len(chunks)} chunks from {filename}")
        return chunks
    print(f"No cache found at {filename}")
    return None

In [49]:
CACHE_FILE = f"{PDF_PATH}_chunks.json"

# 1. Try to load from cache first
final_chunks = load_chunks(CACHE_FILE)

# 2. If no cache, run the full pipeline
if final_chunks is None:
    print("Processing PDF (OCR + Chunking)...")

    # Run your existing chunking function
    final_chunks = to_unstructured_hierarchical_chunks(
        results,
        doc_id=PDF_PATH,
        batch_size=BATCH_SIZE,
        total_pages=len(images),
    )
    
    # Save to cache for next time
    save_chunks(final_chunks, CACHE_FILE)
else:
    print("Skipping OCR and chunking, using cached data.")

Successfully loaded 46 chunks from ./benchmark_pdfs/1706.03762.pdf_chunks.json
Skipping OCR and chunking, using cached data.


## 

In [50]:
import subprocess
import requests
import time
import os

# check if qdrant responds on its default port
try:
    requests.get("http://localhost:6333")
    print("Qdrant is already running!")
except requests.exceptions.ConnectionError:
    print("Qdrant is not running. Attempting to start Docker container...")
    
    # Try to start the existing container first
    result = subprocess.run(["docker", "start", "qdrant_db"], capture_output=True, text=True)
    
    if result.returncode == 0:
         print("Existing Qdrant container started.")
    else:
        print("Container 'qdrant_db' not found or failed to start. Creating a new one...")
        # Fix the path formatting for Docker volume mount
        cwd = os.path.abspath(os.getcwd()).replace('\\', '/')
        volume_mount = f"{cwd}/qdrant_storage:/qdrant/storage"
        
        run_result = subprocess.run([
            "docker", "run", "-d", "-p", "6333:6333", "-p", "6334:6334",
            "-v", volume_mount, 
            "--name", "qdrant_db", "qdrant/qdrant"
        ], capture_output=True, text=True)
        
        if run_result.returncode != 0:
            print(f"ERROR starting container: {run_result.stderr}")
        else:
             print("New Qdrant container created and started.")
        
    # Wait a bit longer for initialization
    print("Waiting for Qdrant to initialize...")
    time.sleep(5)
    
    # Final check
    try:
        requests.get("http://localhost:6333")
        print("Qdrant successfully started and responding!")
    except requests.exceptions.ConnectionError:
        print("WARNING: Qdrant still not responding. Check Docker desktop or run 'docker logs qdrant_db' in your terminal.")

Qdrant is not running. Attempting to start Docker container...
Container 'qdrant_db' not found or failed to start. Creating a new one...
New Qdrant container created and started.
Waiting for Qdrant to initialize...
Qdrant successfully started and responding!


In [51]:
import qdrant_client
from qdrant_client.models import SparseVectorParams
from qdrant_client.models import Distance, VectorParams
from langchain_community.vectorstores import Qdrant
from langchain_community.embeddings import LlamaCppEmbeddings

collection_name = "pdf_chunks"
url = "http://localhost:6333"

qdrant = qdrant_client.QdrantClient(url=url)

ModuleNotFoundError: No module named 'qdrant_client'

In [ ]:
VECTOR_DIMENSION = 4096
qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": VectorParams(size=VECTOR_DIMENSION, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

C:\Users\calla\AppData\Local\Temp\ipykernel_3556\3427014447.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

### Loading embeding

In [ ]:
import os
import ctypes

# define cuda path
cuda_bin = r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0\bin\x64"

# 1 force path in os environment for c++ underlying loadlibrary calls
os.environ["PATH"] = cuda_bin + os.pathsep + os.environ.get("PATH", "")

# 2 force python dll directory
if os.path.exists(cuda_bin):
    os.add_dll_directory(cuda_bin)

# 3 test raw dll load to catch the exact missing dependency
dll_path = r"C:\Users\calla\Documents\AI projects\chatbot\mas\code\.venv\Lib\site-packages\llama_cpp\lib\llama.dll"
try:
    ctypes.CDLL(dll_path)
    print("raw dll loaded successfully")
except Exception as e:
    print(f"raw dll load failed: {e}")

# 4 load langchain model
from langchain_community.embeddings.llamacpp import LlamaCppEmbeddings

model_path = r"C:\Users\calla\Documents\AI projects\chatbot\mas\code\chunk\qwen_embedding\qwen-embed-8B.gguf"

# RTX 5090: larger token batch improves dense embedding throughput.
embeddings = LlamaCppEmbeddings(
    model_path=model_path,
    n_gpu_layers=-1,
    n_batch=1024,
    n_ctx=8192,
    verbose=False,
)

print("model loaded successfully")

raw dll loaded successfully


llama_context: n_ctx_per_seq (8192) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


model loaded successfully


### Loading sparse

In [ ]:
from fastembed import SparseTextEmbedding

fastembed_model_id = "prithivida/Splade_PP_en_v1"

# path to the folder containing your downloaded hf model files
# make sure the onnx files and tokenizer configs are in here
local_splade_path = r"C:\Users\calla\Documents\AI projects\chatbot\mas\code\chunk\splade_local"

# load splade using fastembed
# batch size 512 is easy for an rtx 5090
# cuda execution provider forces it onto the gpu
sparse_embedder = SparseTextEmbedding(
    model_name=fastembed_model_id,
    cache_dir=local_splade_path,
    local_files_only=True,
    providers=["CUDAExecutionProvider"],
    batch_size=128
)

# quick test to ensure it works
texts = ["test document for sparse embedding"]
sparse_vectors = list(sparse_embedder.embed(texts))

# print first vector indices and values
print(sparse_vectors[0])

SparseEmbedding(values=array([0.12816659, 0.56002009, 0.26966175, 0.61802816, 2.28203988,
       0.06547009, 0.22684972, 0.8429935 , 1.09188557, 0.93930495,
       0.31223774, 0.57224894, 0.0927739 , 1.71002758, 0.9207651 ,
       0.2465439 , 1.98386526, 1.77601576, 1.12569249, 0.20433615,
       0.1845513 , 2.31647396]), indices=array([ 1012,  2000,  2005,  2793,  3231,  4007,  4289,  4638,  4667,
        5371,  5491,  5604,  5732,  6254,  6845,  6994,  7861,  8270,
        9742, 12827, 19701, 20288]))


In [ ]:
clear_memory()


Nettoyage de la VRAM en cours...
VRAM libérée avec succès ! Prêt pour le chargement des modèles d'embedding.


## Add vectors

In [ ]:
import uuid
import numpy as np
from tqdm import tqdm
from qdrant_client.models import PointStruct

def batch_insert_to_qdrant(chunks, client, collection_name, dense_embedder, sparse_embedder, batch_size=128):
    """
    Embeds texts and inserts them into Qdrant with both dense and sparse vectors.
    Includes a sub-batching safety net for Llama.cpp to prevent decode crashes.
    """
    total_chunks = len(chunks)
    
    # Qdrant/FastEmbed run perfectly with large batch sizes (128)
    for i in tqdm(range(0, total_chunks, batch_size), desc="Inserting into Qdrant"):
        batch = chunks[i : i + batch_size]
        texts = [chunk["text"] for chunk in batch]
        
        # 1. Generate Dense Embeddings (one text at a time to avoid KV cache overflow)
        # llama_decode -1 = total tokens across all sequences exceeds n_ctx; embed 1 by 1 to be safe.
        dense_vectors = []
        for text in texts:
            dense_vectors.extend(dense_embedder.embed_documents([text]))
        
        # 2. Generate Sparse Embeddings (FastEmbed handles the full 128 list natively)
        sparse_vectors = list(sparse_embedder.embed(texts))
        
        points = []
        for j, chunk in enumerate(batch):
            # Qdrant requires UUIDs or integers. Hash the string ID into a UUID5.
            point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk["id"]))
            
            # Prepare the point payload (metadata + the raw text)
            payload = chunk.get("metadata", {})
            payload["text"] = chunk["text"]
            payload["original_id"] = chunk["id"]
            
            # Construct the point
            points.append(
                PointStruct(
                    id=point_id,
                    payload=payload,
                    vector={
                        "dense": dense_vectors[j],
                        "sparse": {
                            "indices": sparse_vectors[j].indices.tolist(),
                            "values": sparse_vectors[j].values.tolist(),
                        }
                    }
                )
            )
            
        # 3. Upload batch to Qdrant
        client.upload_points(
            collection_name=collection_name,
            points=points,
            wait=True
        )

# --- Execution ---
batch_insert_to_qdrant(
    chunks=final_chunks,
    client=qdrant,
    collection_name=collection_name,
    dense_embedder=embeddings,
    sparse_embedder=sparse_embedder,
    batch_size=20
)

Inserting into Qdrant:   0%|          | 0/1 [00:00<?, ?it/s]init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
Inserting into Qdrant: 100%|██████████| 1/1 [00:00<00:00,  1.24it/s]


In [ ]:
# --- Verification: check vectors inserted in Qdrant ---
collection_info = qdrant.get_collection(collection_name=collection_name)
total_points = collection_info.points_count

print(f"Collection : '{collection_name}'")
print(f"Nombre de vecteurs insérés : {total_points}")
print(f"Status : {collection_info.status}")

# Sample a few points to confirm payload + vectors are present
sample = qdrant.scroll(
    collection_name=collection_name,
    limit=3,
    with_payload=True,
    with_vectors=True
)

print(f"\n--- Exemple de {len(sample[0])} points ---")
for point in sample[0]:
    dense_dim = len(point.vector["dense"]) if point.vector and "dense" in point.vector else "N/A"
    sparse_nnz = len(point.vector["sparse"].indices) if point.vector and "sparse" in point.vector else "N/A"
    print(f"\nID       : {point.id}")
    print(f"Dense    : {dense_dim} dimensions")
    print(f"Sparse   : {sparse_nnz} valeurs non nulles")
    print(f"Payload  : {list(point.payload.keys())}")

Collection : 'pdf_chunks'
Nombre de vecteurs insérés : 7
Status : green

--- Exemple de 3 points ---

ID       : 0059fa50-f18b-588e-b01d-d3ab18e78248
Dense    : 4096 dimensions
Sparse   : 92 valeurs non nulles
Payload  : ['doc_id', 'chunk', 'section_title', 'section_path', 'section_depth', 'page_start', 'page_end', 'page_granularity', 'chunk_type', 'element_types', 'has_table', 'has_list', 'has_image', 'char_len', 'word_count', 'approx_tokens', 'total_chunks', 'position_ratio', 'prev_chunk_id', 'next_chunk_id', 'text', 'original_id']

ID       : 0260db80-56a7-5934-88a2-f0c6a0635459
Dense    : 4096 dimensions
Sparse   : 124 valeurs non nulles
Payload  : ['doc_id', 'chunk', 'section_title', 'section_path', 'section_depth', 'page_start', 'page_end', 'page_granularity', 'chunk_type', 'element_types', 'has_table', 'has_list', 'has_image', 'char_len', 'word_count', 'approx_tokens', 'total_chunks', 'position_ratio', 'prev_chunk_id', 'next_chunk_id', 'text', 'original_id']

ID       : 2b8e2669